## 1. Setup

### Libraries

In [1]:
# weather data 
import meteostat as ms

# from src
from src.spark_config import initialize_spark # spark session
from src.helper_functions import get_coordinates, get_paths 

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract, min, max, year
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

# other libraries
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime, timedelta

# specify java version
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

### Initialize

In [2]:
# set spark session
spark = initialize_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 20:55:23 WARN Utils: Your hostname, LaptopEHM, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/26 20:55:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 20:55:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.3.0
Application ID: local-1795722925216
Spark UI: http://10.255.255.254:4040


### Define Data Paths

In [3]:
# paths for statsbomb data
paths = get_paths()

## 2. Matches

### Load and Filter La Liga Matches

In [4]:
matches_df = spark.read.parquet("data/bronze/statsbomb/matches")
laliga_matches = matches_df.filter(
    col("competition.competition_name") == "La Liga")

### Validation Rules

In [35]:
invalid_match = (
    col("match_id").isNull() |
    col("kick_off").isNull() |
    col("stadium.name").isNull() |
    col("season.season_name").isNull() |
    to_date(col("match_date")).isNull() |
    col("home_score").isNull() |
    col("away_score").isNull() |
    (col("home_score") < 0) |
    (col("away_score") < 0)
)

duplicate_match_ids = laliga_matches.groupBy("match_id").count().filter(col("count") > 1).select("match_id")

### Data Quality Before Cleaning

In [36]:
# data quality of matches before cleaning
print(f"Number of matches in La Liga (before cleaning): ", laliga_matches.count())
print(f"Number of unique matches in La Liga (before cleaning): ", laliga_matches.select("match_id").distinct().count())
print(f"Duplicate match ids (before cleaning): ", duplicate_match_ids.count())
print(f"Invalid matches (before cleaning): ", laliga_matches.filter(invalid_match).count())

Number of matches in La Liga (before cleaning):  867
Number of unique matches in La Liga (before cleaning):  867
Duplicate match ids (before cleaning):  0
Invalid matches (before cleaning):  0


### Validation and Quarantine

In [37]:
quarantine_matches = laliga_matches.filter(invalid_match)

valid_matches = laliga_matches.filter(~invalid_match)

duplicate_matches = valid_matches.join(
    duplicate_match_ids, on="match_id", how="inner"
)

quarantine_matches = quarantine_matches.unionByName(duplicate_matches)

valid_matches = valid_matches.join(
    duplicate_match_ids, on="match_id", how="left_anti"
)

print(f"Quarantined matches: ", quarantine_matches.count())
print(f"Valid matches: ", valid_matches.count())

Quarantined matches:  0
Valid matches:  867


### Transform Matches

In [33]:
# forslag til første utdrag av data
silver_matches = valid_matches.select(
        col("match_id"),
        col("kick_off"),
        col("stadium.name").alias("stadium"),
        col("home_team.home_team_name").alias("home_team"),
        col("away_team.away_team_name").alias("away_team"),
        col("home_team.home_team_id").alias("home_team_id"),
        col("away_team.away_team_id").alias("away_team_id"),
        col("home_score"), # sånn at vi kan se på om hjemmelaget spiller bedre i ulike værforhold?
        col("away_score"),
        col("season.season_name").alias("season"),
        to_date("match_date").alias("match_date")
        )

silver_matches.show(5)
silver_matches.printSchema()

+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
|match_id|    kick_off|             stadium|           home_team|     away_team|home_team_id|away_team_id|home_score|away_score|   season|match_date|
+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
| 3825739|17:00:00.000|            Bernabéu|         Real Madrid|Sporting Gijón|         220|        1041|         5|         1|2015/2016|2016-01-17|
| 3825848|20:00:00.000|Estadio Ciudad de...|          Levante UD|         Eibar|         221|         322|         2|         2|2015/2016|2015-09-23|
| 3825895|22:00:00.000|Estadio de Gran C...|          Las Palmas|       Sevilla|         208|         213|         2|         0|2015/2016|2015-09-23|
| 3825894|18:15:00.000|Estadio Abanca-Ri...|RC Deportivo La C...|        Getafe|         219|       

### Data Quality After Cleaning

In [ ]:
# data quality of matches after cleaning
print(f"Number of matches in La Liga (after cleaning): ", silver_matches.count())
print(f"Number of unique matches in La Liga (after cleaning): ", silver_matches.select("match_id").distinct().count())
print(f"Invalid matches (after cleaning): ", valid_matches.filter(invalid_match).count())

Number of matches in La Liga (after cleaning):  867
Number of unique matches in La Liga (after cleaning):  867
Invalid matches (after cleaning):  0


In [9]:
invalid_matches_after = silver_matches.filter(
    col("match_id").isNull() |
    col("kick_off").isNull() |
    col("stadium").isNull() |
    col("season").isNull() |
    col("match_date").isNull()
).count()

invalid_scores_after = silver_matches.filter(
    (col("home_score") < 0) |
    (col("away_score") < 0)).count()

print(f"Invalid matches (after cleaning): ", invalid_matches_after)
print(f"Invalid scores (after cleaning): ", invalid_scores_after)

Invalid matches (after cleaning):  0
Invalid scores (after cleaning):  0


In [10]:
matches_df.printSchema()

root
 |-- away_score: long (nullable = true)
 |-- away_team: struct (nullable = true)
 |    |-- away_team_gender: string (nullable = true)
 |    |-- away_team_group: string (nullable = true)
 |    |-- away_team_id: long (nullable = true)
 |    |-- away_team_name: string (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- managers: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- country: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- dob: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nickname: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- competition_id: long (nullable = true)
 |    |-- competition_nam

## 3. Events

### Load and Filter La Liga Events

In [11]:
events_df = spark.read.parquet("data/bronze/statsbomb/events")

laliga_events = events_df.join(
    laliga_matches.select("match_id"),
    on="match_id",
    how="inner"
)

### Data Quality Before Cleaning

In [12]:
print(f"Number of events in La Liga (before cleaning): ", laliga_events.count())
print(f"Number of unique events in La Liga (before cleaning): ", laliga_events.select("id").distinct().count())

Number of events in La Liga (before cleaning):  3132716


Number of unique events in La Liga (before cleaning):  3132716


In [13]:
invalid_events_before = laliga_events.filter(
    col("match_id").isNull() |
    col("id").isNull() |
    (col("type.name") == "Pass") & col("pass").isNull()
).count()

print(f"Invalid events (before cleaning): ", invalid_events_before)

Invalid events (before cleaning):  0


### Validation and Quarantine

### Transform Events

In [16]:
silver_events = laliga_events.select(
    col("match_id"),
    col("id").alias("event_id"),
    col("type.name").alias("event_type"),
    col("pass"),
    col("team.name").alias("team"),
    col("player.name").alias("player"),
    col("shot"),
    col("foul_committed")
    )

In [17]:
silver_events.printSchema()
silver_events.show(5)

root
 |-- match_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- pass: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- angle: double (nullable = true)
 |    |-- assisted_shot_id: string (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- cross: boolean (nullable = true)
 |    |-- cut_back: boolean (nullable = true)
 |    |-- deflected: boolean (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- goal_assist: boolean (nullable = true)
 |    |-- height: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- inswinging: boolean (nullable = true)
 |    |-- length: double (nullable = true)
 |    |-- miscommunication: boolean (nullable = true)


### Data Quality After Cleaning

In [18]:
print(f"Number of events in La Liga (after cleaning): ", silver_events.count())
print(f"Number of unique events in La Liga (after cleaning): ", silver_events.select("event_id").distinct().count())

Number of events in La Liga (after cleaning):  3132716


Number of unique events in La Liga (after cleaning):  3132716


In [19]:
invalid_events_after = silver_events.filter(
    col("match_id").isNull() |
    col("event_id").isNull() |
    (col("event_type") == "Pass") & col("pass").isNull()
).count()

print(f"Invalid events (after cleaning): ", invalid_events_after)

Invalid events (after cleaning):  0


In [20]:
events_df.printSchema()

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearance: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- head: boolean (nullable = true)
 |    |-- left_f

## 4. Weather 

In [21]:
# get number of matches in each season
silver_matches.groupBy("season").count().orderBy("season").show()

+---------+-----+
|   season|count|
+---------+-----+
|1973/1974|    1|
|2004/2005|    7|
|2005/2006|   17|
|2006/2007|   26|
|2007/2008|   27|
|2008/2009|   31|
|2009/2010|   35|
|2010/2011|   33|
|2011/2012|   37|
|2012/2013|   32|
|2013/2014|   31|
|2014/2015|   38|
|2015/2016|  380|
|2016/2017|   34|
|2017/2018|   36|
|2018/2019|   34|
|2019/2020|   33|
|2020/2021|   35|
+---------+-----+



In [22]:
# get number of matches by each year
silver_matches.groupBy(
    year("match_date").alias("year")
).count().orderBy("year").show(truncate=False)

+----+-----+
|year|count|
+----+-----+
|1974|1    |
|2004|5    |
|2005|12   |
|2006|16   |
|2007|31   |
|2008|27   |
|2009|30   |
|2010|36   |
|2011|35   |
|2012|38   |
|2013|26   |
|2014|36   |
|2015|191  |
|2016|224  |
|2017|38   |
|2018|34   |
|2019|32   |
|2020|34   |
|2021|21   |
+----+-----+



In [23]:
# stadiums we need to find weather stations for
silver_matches.select("stadium").distinct().orderBy("stadium").show(50, truncate=False)

+---------------------------------------------------------+
|stadium                                                  |
+---------------------------------------------------------+
|Abanca-Balaídos                                          |
|Bernabéu                                                 |
|Campos de Sport de El Sardinero                          |
|Coliseum Alfonso Pérez                                   |
|Estadi Municipal de Montilivi                            |
|Estadi Olímpic Lluís Companys                            |
|Estadio Abanca-Riazor                                    |
|Estadio Alfredo Di Stéfano                               |
|Estadio Benito Villamarín                                |
|Estadio Carlos Belmonte                                  |
|Estadio Ciudad de Valencia                               |
|Estadio El Alcoraz                                       |
|Estadio El Sadar                                         |
|Estadio Enrique Roca de Murcia         

### Load Weather Data

### Data Quality Before Cleaning

### Validation and Quarantine

### Transform Weather Data

### Data Quality After Cleaning

### Join Weather With Matches

## 5. Write Silver Data To Parquet

In [45]:
from src.helper_functions import get_coordinates

# Known stadium
print(get_coordinates("Bernabéu", "Spain"))

# Unknown stadium
print(get_coordinates("Old Trafford", "England"))

# Non-existent stadium
print(get_coordinates("Nonexistent Stadium XYZ123"))

(40.453044, -3.688197)
(53.4631096, -2.2913865)


No coordinates found for Nonexistent Stadium XYZ123


None


In [25]:
from geopy.geocoders import Nominatim

geo_locator = Nominatim(user_agent="dat535-football-stats")

location = geo_locator.geocode(
    "Stamford Bridge, England",
    addressdetails=True
)

print(location.raw)

{'place_id': 281557342, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 5986169, 'lat': '51.4816869', 'lon': '-0.1910336', 'class': 'leisure', 'type': 'stadium', 'place_rank': 30, 'importance': 0.6028355034306307, 'addresstype': 'leisure', 'name': 'Stamford Bridge', 'display_name': 'Stamford Bridge, Britannia Gate, Walham Green, Fulham, London Borough of Hammersmith and Fulham, Greater London, England, SW6 1HD, United Kingdom', 'address': {'leisure': 'Stamford Bridge', 'road': 'Britannia Gate', 'neighbourhood': 'Walham Green', 'suburb': 'Fulham', 'city_district': 'London Borough of Hammersmith and Fulham', 'ISO3166-2-lvl8': 'GB-HMF', 'city': 'Greater London', 'state': 'England', 'ISO3166-2-lvl4': 'GB-ENG', 'postcode': 'SW6 1HD', 'country': 'United Kingdom', 'country_code': 'gb'}, 'boundingbox': ['51.4807699', '51.4826215', '-0.1924468', '-0.1896121']}
